# Comparativa de modelos - Regresión

Objetivo: comparar dos versiones de cada modelo de regresión usando validación cruzada y un conjunto holdout sobre Housing Prices.

## 1) Librerías

In [ ]:
!pip install pandas matplotlib seaborn scikit-learn


In [ ]:
# Importamos sklearn y las herramientas necesarias para repetir el mismo flujo que en clasificación.
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import get_scorer

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor


## 2) Cargar datos

In [ ]:
data = pd.read_csv("housing.csv")
data.head()


In [ ]:
data.info()


## 3) Preparar datos

In [ ]:
variables_a_descartar = []
variables_categoricas = ["CHAS"]
variables_numericas = [
    "CRIM", "ZN", "INDUS", "NOX", "RM", "AGE",
    "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT"
]
variable_dependiente = "MEDV"

X = data.drop([variable_dependiente] + variables_a_descartar, axis=1)
y = data[variable_dependiente].astype(float)

print(X.head())
print(y.head())


## 4) Definir holdout y métricas

In [ ]:
random_state = 42
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=random_state
)

# En sklearn varios errores se reportan con signo negativo para que "más alto" siga significando "mejor".
metricas_disponibles = ["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error", "neg_mean_absolute_percentage_error"]
metricas_seleccionadas = ["r2", "neg_root_mean_squared_error"]

for metrica in metricas_seleccionadas:
    if metrica not in metricas_disponibles:
        raise ValueError(f"La métrica {metrica} no está en la lista disponible")

scoring = {metrica: metrica for metrica in metricas_seleccionadas}
cv = KFold(n_splits=5, shuffle=True, random_state=random_state)


## 5) Grilla de comparación

In [ ]:
preprocesamiento = ColumnTransformer([
    (
        "numerico",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        variables_numericas,
    ),
    (
        "categorico",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        variables_categoricas,
    ),
])

comparison_grid = [
    {"modelo": "KNN", "version": "k=5", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", KNeighborsRegressor(n_neighbors=5))])},
    {"modelo": "KNN", "version": "k=11", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", KNeighborsRegressor(n_neighbors=11))])},
    {"modelo": "Regresión lineal", "version": "OLS", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", LinearRegression())])},
    {"modelo": "Regresión lineal", "version": "Ridge(alpha=1)", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", Ridge(alpha=1.0, random_state=random_state))])},
    {"modelo": "Árbol de decisión", "version": "max_depth=4", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", DecisionTreeRegressor(max_depth=4, random_state=random_state))])},
    {"modelo": "Árbol de decisión", "version": "max_depth=8", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", DecisionTreeRegressor(max_depth=8, random_state=random_state))])},
    {"modelo": "Random forest", "version": "200 árboles", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", RandomForestRegressor(n_estimators=200, random_state=random_state))])},
    {"modelo": "Random forest", "version": "400 árboles", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", RandomForestRegressor(n_estimators=400, random_state=random_state))])},
    {"modelo": "SVM", "version": "C=1", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", SVR(C=1.0, kernel="rbf", epsilon=0.1))])},
    {"modelo": "SVM", "version": "C=10", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", SVR(C=10.0, kernel="rbf", epsilon=0.1))])},
    {"modelo": "Red neuronal", "version": "(32,)", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", MLPRegressor(hidden_layer_sizes=(32,), max_iter=1000, random_state=random_state))])},
    {"modelo": "Red neuronal", "version": "(64, 32)", "pipeline": Pipeline([("preprocesamiento", preprocesamiento), ("modelo", MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=random_state))])},
]

pd.DataFrame([{"modelo": item["modelo"], "version": item["version"]} for item in comparison_grid])


## 6) CV out-of-set + holdout

In [ ]:
resultados = []

for item in comparison_grid:
    pipeline = item["pipeline"]

    cv_scores = cross_validate(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=None,
    )

    pipeline.fit(X_train, y_train)

    fila = {
        "modelo": item["modelo"],
        "version": item["version"],
    }

    for metrica in metricas_seleccionadas:
        cv_mean = cv_scores[f"test_{metrica}"].mean()
        cv_std = cv_scores[f"test_{metrica}"].std()
        holdout_score = get_scorer(metrica)(pipeline, X_holdout, y_holdout)

        fila[f"cv_mean_{metrica}"] = cv_mean
        fila[f"cv_std_{metrica}"] = cv_std
        fila[f"holdout_{metrica}"] = holdout_score
        fila[f"gap_{metrica}"] = holdout_score - cv_mean

    resultados.append(fila)

resultados_df = pd.DataFrame(resultados)
resultados_df


## 7) Comparar resultados

In [ ]:
metrica_principal = metricas_seleccionadas[0]
columnas = ["modelo", "version"]
for metrica in metricas_seleccionadas:
    columnas.extend([f"cv_mean_{metrica}", f"cv_std_{metrica}", f"holdout_{metrica}", f"gap_{metrica}"])

comparacion = resultados_df[columnas].sort_values(by=f"cv_mean_{metrica_principal}", ascending=False)
comparacion


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=comparacion, x="modelo", y=f"cv_mean_{metrica_principal}", hue="version")
plt.title(f"Comparación por CV según {metrica_principal}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
